# Retos del workshop RAG · notebook de arranque

Este notebook trae el punto de partida de algunos retos. Los bloques marcados con `TODO` son tuyos. Copia este archivo a `notebooks/04_challenges.ipynb` en tu repo.

> **Entrega · Daniel Muñeton (SI3003).** Resuelvo los `TODO` de los retos **C1**, **C4** y **C6**. Los retos C2, C3 y C5 estan en el notebook `03`.

## C1 · HNSW contra fuerza bruta con FAISS

Instala FAISS y ejecuta la línea base. Después completa los TODO.

In [ ]:
%pip install -q faiss-cpu matplotlib

Linea base con FAISS, parametrizada por `n` para reutilizarla en el TODO 2.

In [ ]:
import time
import faiss
import numpy as np

def experimento(n, d=64, nq=200, k=10, seed=42):
    rng = np.random.default_rng(seed)
    xb = rng.random((n, d), dtype="float32")
    xq = rng.random((nq, d), dtype="float32")

    flat = faiss.IndexFlatL2(d)
    flat.add(xb)
    t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000

    def recall_at_k(I, I_true):
        return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

    rows = []
    for M in (8, 16, 32):
        index = faiss.IndexHNSWFlat(d, M)
        index.hnsw.efConstruction = 100
        t0 = time.time(); index.add(xb); build_s = time.time() - t0
        for ef in (10, 16, 32, 64, 128, 256):
            index.hnsw.efSearch = ef
            t0 = time.time(); _, I = index.search(xq, k); ms = (time.time() - t0) / nq * 1000
            rows.append((M, ef, recall_at_k(I, I_true), ms, build_s))
    return t_flat, rows

t_flat, rows = experimento(20_000)
print(f"exacto: {t_flat:.3f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@10={rec:.3f}  {ms:.3f} ms/q  (build {b:.1f}s)")

### TODO 1 (resuelto) · Grafica recall vs efSearch, una curva por M

In [ ]:
import matplotlib.pyplot as plt

def graficar(rows, titulo):
    plt.figure(figsize=(7, 4))
    for M in sorted({r[0] for r in rows}):
        efs  = [r[1] for r in rows if r[0] == M]
        recs = [r[2] for r in rows if r[0] == M]
        plt.plot(efs, recs, marker="o", label=f"M={M}")
    plt.xscale("log")
    plt.xlabel("efSearch (escala log)")
    plt.ylabel("recall@10")
    plt.title(titulo)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

graficar(rows, "HNSW · recall@10 vs efSearch (n=20k)")

### TODO 2 (resuelto) · Barrido de n y punto de cruce

Comparo la latencia del mejor HNSW con recall ≥ 0.95 contra la busqueda exacta para n crecientes. El punto de cruce es el n donde HNSW empieza a ser mas rapido.

> n = 500_000 puede tardar varios minutos en construir.

In [ ]:
resumen = []
for n in (20_000, 100_000, 500_000):
    t_flat_n, rows_n = experimento(n)
    buenos = [r for r in rows_n if r[2] >= 0.95]
    if buenos:
        mejor = min(buenos, key=lambda r: r[3])
        resumen.append((n, t_flat_n, mejor[3]))
        print(f"n={n:>7}  exacto {t_flat_n:7.3f} ms/q  |  HNSW {mejor[3]:7.3f} ms/q "
              f"(M={mejor[0]}, ef={mejor[1]}, recall={mejor[2]:.3f})  "
              f"-> HNSW {'MAS RAPIDO' if mejor[3] < t_flat_n else 'mas lento'}")
    else:
        print(f"n={n}: ningun HNSW alcanzo recall>=0.95")

print("\nResumen (n, exacto_ms, hnsw_ms):", [(r[0], round(r[1],3), round(r[2],3)) for r in resumen])
print("Conclusiones en RESPUESTAS.md (C1).")

### TODO 3 (resuelto) · Conclusiones en `RESPUESTAS.md` (seccion C1).

In [ ]:
import time
import faiss
import numpy as np

d, n, nq, k = 64, 20_000, 200, 10
rng = np.random.default_rng(42)
xb = rng.random((n, d), dtype="float32")
xq = rng.random((nq, d), dtype="float32")

flat = faiss.IndexFlatL2(d)
flat.add(xb)
t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000

def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

rows = []
for M in (8, 16, 32):
    index = faiss.IndexHNSWFlat(d, M)
    index.hnsw.efConstruction = 100
    t0 = time.time(); index.add(xb); build_s = time.time() - t0
    for ef in (10, 16, 32, 64, 128, 256):
        index.hnsw.efSearch = ef
        t0 = time.time(); _, I = index.search(xq, k); ms = (time.time() - t0) / nq * 1000
        rows.append((M, ef, recall_at_k(I, I_true), ms, build_s))

print(f"exacto: {t_flat:.3f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@{k}={rec:.3f}  {ms:.3f} ms/q  (build {b:.1f}s)")

In [ ]:
# TODO 1: grafica recall vs efSearch (una curva por M) con matplotlib.
# TODO 2: repite el experimento con n = 100_000 y n = 500_000. ¿Desde qué n HNSW supera al exacto?
# TODO 3: escribe tus conclusiones en RESPUESTAS.md

### TODO (resuelto) · `hit@3` y `hit@5` con chunk_size 400, 900 y 1500

Uso `eval/preguntas.csv` (ampliado a 10 filas). Requiere `attention_text`, `rag_text` y `embeddings` del notebook 03 (ejecuta antes sus Partes 0-3).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

filas = cargar_preguntas()
print(f"Preguntas cargadas: {len(filas)}")

def indexar(docs, nombre):
    dim = len(embeddings.embed_query("dimension probe"))
    client = QdrantClient(":memory:")
    client.create_collection(collection_name=nombre,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
    vs = QdrantVectorStore(client=client, collection_name=nombre, embedding=embeddings)
    vs.add_documents(docs)
    return vs

FUENTES = [
    (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
    (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf"),
]
tabla = []
for cs in (400, 900, 1500):
    docs = []
    sp = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=min(120, cs//8),
                                        separators=["\n\n", "\n", ".", " "])
    for txt, titulo, src in FUENTES:
        for idx, ch in enumerate(sp.split_text(txt)):
            if len(ch.strip()) >= 30:
                docs.append(Document(page_content=ch, metadata={"title": titulo, "source": src, "chunk_index": idx}))
    vs = indexar(docs, f"eval_cs_{cs}")
    r3 = vs.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    r5 = vs.as_retriever(search_type="similarity", search_kwargs={"k": 5})
    h3, h5 = hit_at_k(r3, filas), hit_at_k(r5, filas)
    tabla.append((cs, len(docs), h3, h5))
    print(f"chunk_size={cs:>4}  chunks={len(docs):>4}  hit@3={h3:.2f}  hit@5={h5:.2f}")

print("\n| chunk_size | chunks | hit@3 | hit@5 |")
print("|---|---|---|---|")
for cs, nc, h3, h5 in tabla:
    print(f"| {cs} | {nc} | {h3:.2f} | {h5:.2f} |")

## C4 · Evaluación automática de la recuperación

Necesitas las variables del notebook 03 (`retriever`, `all_docs`, …) o reconstruirlas aquí.

In [ ]:
import csv

def cargar_preguntas(ruta="eval/preguntas.csv"):
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def hit_at_k(retriever, filas):
    hits = 0
    for r in filas:
        docs = retriever.invoke(r["pregunta"])
        ok = any(
            r["palabra_clave"].lower() in d.page_content.lower()
            and d.metadata["source"] == r["fuente_esperada"]
            for d in docs
        )
        hits += ok
    return hits / len(filas)

# TODO: amplía eval/preguntas.csv a 10 filas, calcula hit@3 y hit@5 con chunk_size 400, 900 y 1500.

### TODO (resuelto) · Indexar/reutilizar midiendo el tiempo

Requiere `all_docs` y `embeddings` del notebook 03. Primera ejecucion: indexa y mide; tras reiniciar el runtime: reutiliza sin reindexar.

In [ ]:
import time
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COL = "academic_papers_nvidia"
t0 = time.time()
if not qdrant_client.collection_exists(COL):
    dim = len(embeddings.embed_query("dimension probe"))
    qdrant_client.create_collection(collection_name=COL,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
    vs = QdrantVectorStore(client=qdrant_client, collection_name=COL, embedding=embeddings)
    vs.add_documents(all_docs)
    print(f"Indexado desde cero en {time.time()-t0:.2f} s")
else:
    vs = QdrantVectorStore(client=qdrant_client, collection_name=COL, embedding=embeddings)
    print(f"Reutilizada sin reindexar en {time.time()-t0:.2f} s")

docs = vs.as_retriever(search_kwargs={"k": 3}).invoke("What is multi-head attention?")
print("Chunks recuperados desde disco:", len(docs))
# Reinicia el runtime, ejecuta solo lo necesario para tener all_docs/embeddings,
# y vuelve a correr: debe decir "Reutilizada sin reindexar".